# SurrealEngine 1.x Features (SurrealKV)

This notebook demonstrates the 1.x feature set on an embedded `surrealkv://` connection:

1. Reference delete policies (`REFERENCE ON DELETE ...`) — gated (2.x experimental capability)
2. Temporal query API (`version_at`, `version_at_raw`)
3. Changefeed replay helper (`show_changes`)
4. Duration ergonomics for `timeout(...)`
5. Additive full-text helpers (`search_and`, `search_or`, score/highlight projections)


In [ ]:
import datetime
from surrealengine import (
    Document,
    StringField,
    ReferenceField,
    create_connection,
)


In [ ]:
# Embedded SurrealKV connection
conn = create_connection(
    url="surrealkv://data/v1x_demo",
    namespace="demo",
    database="v1x_features",
    make_default=True,
)
await conn.connect()
print("Connected to embedded SurrealKV")


## 1) Reference delete policies (2.x capability-gated)

`ReferenceField` now supports native delete behavior via `REFERENCE ON DELETE ...`.

> On SurrealDB 1.x this may fail with: `Experimental capability record_references is not enabled`.
> On SurrealDB 2.x, enable the capability via `--allow-experimental record_references` (or env var equivalent).


In [ ]:
class User(Document):
    name = StringField(required=True)

    class Meta:
        collection = "users_v1x"


class Comment(Document):
    text = StringField(required=True)
    # If a user is deleted, unset comment.author
    author = ReferenceField("User", reference=True, on_delete="UNSET")

    class Meta:
        collection = "comments_v1x"


class AuditLog(Document):
    ref = ReferenceField(
        "Comment",
        on_delete="THEN",
        on_delete_then="UPDATE $this SET archived_ref += $reference, ref = NONE;",
    )

    class Meta:
        collection = "audit_logs_v1x"


await User.create_table()
try:
    await Comment.create_table()
    await AuditLog.create_table()
    print("Tables created with REFERENCE delete policies")
except Exception as exc:
    print("Skipping reference policy DDL in this runtime:")
    print(exc)
    print("Tip: use SurrealDB 2.x + allow-experimental record_references to run this section.")


## 2) Temporal query API (`VERSION`)

`version_at(...)` and `version_at_raw(...)` are now available on querysets.

> Note: `VERSION` support depends on backend/versioning mode and may be experimental depending on SurrealDB version.


In [ ]:
version_ts = datetime.datetime(2026, 3, 31, 8, 0, 0, tzinfo=datetime.timezone.utc)

q1 = User.objects.version_at(version_ts)
q2 = User.objects.version_at_raw("time::now()")

print(q1.get_raw_query())
print(q2.get_raw_query())


## 3) Changefeed replay helper (`show_changes`)

Use the connection helper to replay events from a changefeed cursor.


In [ ]:
# Ensure a changefeed exists for the table
await conn.client.query("DEFINE TABLE users_v1x CHANGEFEED 3d;")

await conn.client.query("CREATE users_v1x:alice SET name = 'Alice';")
await conn.client.query("UPDATE users_v1x:alice SET name = 'Alice A';")

changes = await conn.show_changes("users_v1x", since=0, limit=20)
changes


## 4) Duration ergonomics for timeout

`timeout(...)` accepts strings, `datetime.timedelta`, and SDK `Duration`.


In [ ]:
# timedelta form
q_timeout = User.objects.timeout(datetime.timedelta(seconds=1, milliseconds=250))
print(q_timeout.get_raw_query())

# string form (unchanged)
print(User.objects.timeout("500ms").get_raw_query())


## 5) Additive full-text helpers

These extend existing FTS support without replacing current APIs.


In [ ]:
class Article(Document):
    title = StringField()
    content = StringField()

    class Meta:
        collection = "articles_v1x"
        indexes = [
            {
                "name": "idx_articles_content",
                "fields": ["content"],
                "search": True,
                "bm25": True,
                "highlights": True,
            }
        ]


await Article.create_table()

fts_q = (
    Article.objects
    .search_and("rare personal", "content")
    .with_search_score(reference=1, alias="score")
    .with_search_highlight("<b>", "</b>", reference=1, alias="hl")
    .order_by("score", "DESC")
    .limit(5)
)

print(fts_q.get_raw_query())


## Release Companion Notebooks

- `v1_2_syncmanager_routing.ipynb` (freshness routing)
- `v1_2_context_scoped_sync_manager.ipynb` (request/task scoped routing)
- `v1_2_live_reliability_patterns.ipynb` (LIVE dedicated connection + backpressure)
- `v1_2_replay_checkpoint_recovery.ipynb` (checkpoint/replay recovery)
- `v1_2_search_ergonomics.ipynb` (FTS + vector ergonomic patterns)

`show_changes(...)` now validates table names and accepted `since` formats for safer production use.

In [ ]:
# Optional cleanup
await conn.disconnect()
print("Disconnected")
